In [2]:
# Research Question 1: 
### How do danceability and speechiness levels vary between songs with above-average and below-average Spotify popularity scores among the top 100 songs within the year?

In [4]:
import pandas as pd
import oracledb
import matplotlib.pyplot as plt

In [5]:
#### 1. Oracle DB Connection and SQL File Insertion

In [6]:
dsn = oracledb.makedsn("localhost", 1522, service_name="stu")
connection = oracledb.connect(
    user="ora_david838",
    password="a87164927",
    dsn=dsn
)
cur = connection.cursor()

In [ ]:
#read the SQL file
with open("insert_data_v2.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

#split SQL statements
statements = sql_script.split(";")

#execute each statement
for i, stmt in enumerate(statements, start=1):
    stmt = stmt.strip()
    if stmt:
        try:
            cur.execute(stmt)
        except Exception as e:
            print(f"Skipping error at statement {i}: {e}")

connection.commit()

In [ ]:
#### Dancebility Query 

In [ ]:
#query for danceability
rq4_danceability = """
SELECT 
    popularity_category,
    AVG(danceability) AS avg_danceability
FROM (
    SELECT 
        s.track_id,
        s.danceability,
        s.track_popularity,
        CASE
            WHEN s.track_popularity <= (SELECT AVG(track_popularity) 
                                       FROM (
                                           SELECT track_id, AVG(streams) AS avg_streams
                                           FROM CHARTS
                                           GROUP BY track_id
                                           ORDER BY AVG(streams) DESC
                                           FETCH FIRST 100 ROWS ONLY
                                       ) TOP_100
                                       JOIN SPOTIFY_SONGS ss ON TOP_100.track_id = ss.track_id
                                      ) THEN 'Below Average Popularity'
            ELSE 'Above Average Popularity'
        END AS popularity_category
    FROM (
        SELECT track_id, AVG(streams) AS avg_streams
        FROM CHARTS
        GROUP BY track_id
        ORDER BY AVG(streams) DESC
        FETCH FIRST 100 ROWS ONLY
    ) TOP_100
    JOIN SPOTIFY_SONGS s ON TOP_100.track_id = s.track_id
)
GROUP BY popularity_category
"""

rq4_danceability_df = pd.read_sql(rq4_danceability, connection)
print(rq4_danceability_df.head())

In [1]:
#### Speechiness Query

In [ ]:
#query for speechiness
rq4_speechiness = """
SELECT 
    popularity_category,
    AVG(speechiness) AS avg_speechiness
FROM (
    SELECT 
        s.track_id,
        s.speechiness,
        s.track_popularity,
        CASE
            WHEN s.track_popularity <= (SELECT AVG(track_popularity) 
                                       FROM (
                                           SELECT track_id, AVG(streams) AS avg_streams
                                           FROM CHARTS
                                           GROUP BY track_id
                                           ORDER BY AVG(streams) DESC
                                           FETCH FIRST 100 ROWS ONLY
                                       ) TOP_100
                                       JOIN SPOTIFY_SONGS ss ON TOP_100.track_id = ss.track_id
                                      ) THEN 'Below Average Popularity'
            ELSE 'Above Average Popularity'
        END AS popularity_category
    FROM (
        SELECT track_id, AVG(streams) AS avg_streams
        FROM CHARTS
        GROUP BY track_id
        ORDER BY AVG(streams) DESC
        FETCH FIRST 100 ROWS ONLY
    ) TOP_100
    JOIN SPOTIFY_SONGS s ON TOP_100.track_id = s.track_id
)
GROUP BY popularity_category
"""

rq4_speechiness_df = pd.read_sql(rq4_speechiness, connection)
print(rq4_speechiness_df.head())

In [ ]:
#### Dancebility visualization 

In [ ]:
rq4_danceability_df = rq4_danceability_df.sort_values("POPULARITY_CATEGORY")
rq4_1 = rq4_danceability_df.plot(
    x="POPULARITY_CATEGORY",
    y="AVG_DANCEABILITY",
    kind="bar",
    color=["orange", "blue"],
    legend=False
)

for container in rq4_1.containers:
    rq4_1.bar_label(container, fmt="%.3f")

plt.title("Average Danceability by Popularity Level (Top 100 Streamed Songs)")
plt.xticks(rotation=30)
plt.xlabel("Popularity Category")
plt.ylabel("Average Danceability (0-1)")
plt.ylim(0, 1)  # Set y-axis to 0-1 range
plt.tight_layout()
plt.show()

In [ ]:
#### Speechiness Visualization

In [ ]:
rq4_speechiness_df = rq4_speechiness_df.sort_values("POPULARITY_CATEGORY")
rq4_2 = rq4_speechiness_df.plot(
    x="POPULARITY_CATEGORY",
    y="AVG_SPEECHINESS",
    kind="bar",
    color=["orange", "blue"],
    legend=False
)

for container in rq4_2.containers:
    rq4_2.bar_label(container, fmt="%.3f")

plt.title("Average Speechiness by Popularity Level (Top 100 Streamed Songs)")
plt.xticks(rotation=30)
plt.xlabel("Popularity Category")
plt.ylabel("Average Speechiness (0-1)")
plt.ylim(0, 1)  # Set y-axis to 0-1 range
plt.tight_layout()
plt.show()

In [ ]:
cur.close()
connection.close()